In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['hatch.linewidth'] = 0.2
import numpy as np
import polars as pl

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.file_locations import intermediate_files_location
from src.plot_helpers import make_histogram_plot
from src.ntuple_variables.variables import combined_training_vars
from src.df_helpers import lazy_height

# Data Loading

In [ ]:
# Load the full dataset (lazy)
print("Loading all_df.parquet...")
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
print(f"Total events: {lazy_height(all_df)}")

# Filter: remove overlay files, require valid Enu, exclude run 4a
presel_df = (
    all_df
    .filter(~pl.col("filetype").is_in([
        "isotropic_one_gamma_overlay", "delete_one_gamma_overlay",
        "numuCC_rad_corrected", "NC_coherent_1g_reweighted"
    ]))
    .filter(pl.col("wc_kine_reco_Enu") > 0)
    .filter(pl.col("normalizing_run_period") != "4a")
)

# --- Selection Definitions ---
# Define as Polars expressions so they can be reused on any dataframe (main df and detvar df).

# Erin inclusive 1g selection
erin_1g_sel_expr = (
    (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
    (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
    (pl.col("wc_single_photon_numu_score") > 0.4) &
    (pl.col("wc_single_photon_other_score") > 0.2) &
    (pl.col("wc_single_photon_ncpi0_score") > -0.05) &
    (pl.col("wc_single_photon_nue_score") > -1.0) &
    (pl.col("wc_shw_sp_n_20br1_showers") == 1)
)

# NC pi0 sideband: relaxed numu/other scores, inverted ncpi0 score
erin_ncpi0_sel_expr = (
    (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
    (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
    (pl.col("wc_single_photon_numu_score") > 0.1) &
    (pl.col("wc_single_photon_other_score") > -0.4) &
    (pl.col("wc_single_photon_ncpi0_score") < -0.4) &
    (pl.col("wc_single_photon_nue_score") > -20.0)
)

# Add selection flags
presel_df = presel_df.with_columns([
    erin_1g_sel_expr.cast(pl.Int32).alias("erin_inclusive_1g_sel"),
    erin_ncpi0_sel_expr.cast(pl.Int32).alias("erin_nc_pi0_sideband_flag"),
])

In [ ]:
# Select physics variables (drop training-only variables to save memory)
load_vars = [col for col in presel_df.collect_schema().names() if col not in combined_training_vars]

extra_vars = [
    "wc_kine_reco_Enu",
    "wc_reco_num_protons_35_MeV",
    "wc_reco_backwards_projected_dist",
    "wc_reco_distance_to_boundary",
    "wc_reco_shower_theta",
    "wc_reco_shower_phi",
    "wc_kine_pio_mass",
    "lantern_diphoton_mass",
]
for var in extra_vars:
    if var not in load_vars:
        load_vars.append(var)

presel_merged_df = presel_df.select(load_vars).collect()
print(presel_merged_df.shape)

In [ ]:
# Reweighting systematics
weights_df = pl.read_parquet(f"{intermediate_files_location}/presel_weights_df.parquet")

# Detector variation dataframe (a few columns are absent from the detvar files)
detvar_load_vars = [v for v in load_vars if v not in ["wc_evtTimeNS_cor", "train_test_score", "will_use_for_50_50_training"]]

detvar_df = (
    pl.scan_parquet(f"{intermediate_files_location}/detvar_presel_df_train_vars.parquet")
    .with_columns([
        erin_1g_sel_expr.cast(pl.Int32).alias("erin_inclusive_1g_sel"),
        erin_ncpi0_sel_expr.cast(pl.Int32).alias("erin_nc_pi0_sideband_flag"),
    ])
    .select(detvar_load_vars + ["vartype"])
    .filter(pl.col("normalizing_run_period") != "4a")
    .collect()
)

# Plot Helpers

Common parameters are collected into dicts so each `make_histogram_plot` call only specifies
what's unique to that plot.

In [ ]:
# Default kwargs for all plots with full systematics.
# Produces three figures per call: the main histogram, the full systematic breakdown,
# and the detector-variation overlay + detvar-only fractional uncertainty breakdown.
sys_kwargs = dict(
    weights_df=weights_df,
    use_rw_systematics=True,
    use_detvar_systematics=True,
    num_bootstrap_rounds_detvar=100,
    num_bootstrap_samples_detvar=100,
    plot_sys_breakdown=True,
    plot_det_variations=True,
    plot_detvar_breakdown=True,
    data_type="4b open data",
    include_overflow=False,
    legend_fontsize=10,
    legend_ncol=1,
)

generic_sys_kwargs = dict(
    weights_df=weights_df,
    use_rw_systematics=True,
    use_detvar_systematics=True,
    num_bootstrap_rounds_detvar=100,
    num_bootstrap_samples_detvar=100,
    plot_sys_breakdown=True,
    plot_det_variations=True,
    plot_detvar_breakdown=True,
    data_type="4b open data",
    include_overflow=True,
)

# Blip-based neutron (Nn/0n) split expressions
zero_n_expr = (
    (pl.col("blip_no_shower_cone_no_backtrack_cones_n") <= 2) &
    (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") <= 11)
)
Nn_expr = (
    (pl.col("blip_no_shower_cone_no_backtrack_cones_n") > 2) |
    (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") > 11)
)

# Proton count variables reused across many plots
proton_bins = np.linspace(0, 4, 5)
proton_count_var = "wc_reco_num_protons_35_MeV"
proton_count_display_var = "WC Reconstructed num protons (35 MeV threshold)"
proton_blip_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
proton_blip_display_var = "WC Reconstructed num protons (35 MeV) + backtrack blips"

# Preselection Histogram

In [ ]:
make_histogram_plot(
    pred_and_data_sel_df=presel_merged_df,
    bins=np.linspace(0, 2000, 21),
    var="wc_kine_reco_Enu",
    display_var=r"WC Reconstructed $E_\nu$ (GeV)",
    title="Run 4b WC Inclusive 1g Selection",
    detvar_df=detvar_df,
    selname="generic_presel",    
    **generic_sys_kwargs,
)

# Erin Inclusive 1g Selection

In [ ]:
erin_1g_sel_df = presel_merged_df.filter(pl.col("erin_inclusive_1g_sel") == 1)
erin_1g_sel_detvar_df = detvar_df.filter(pl.col("erin_inclusive_1g_sel") == 1)

make_histogram_plot(
    pred_and_data_sel_df=erin_1g_sel_df,
    bins=proton_bins,
    var=proton_count_var,
    display_var=proton_count_display_var,
    title="Erin Inclusive 1g Selection",
    detvar_df=erin_1g_sel_detvar_df,
    selname="erin_inclusive_1g_sel_new",
    breakdown_type="erin_categories",
    ylim=[0, 50], yticks=np.linspace(0, 50, 11),
    **sys_kwargs,
)

## With CRT Veto

In [ ]:
erin_1g_crt_df = erin_1g_sel_df.filter(pl.col("pandora_crtveto") == 0)
erin_1g_crt_detvar_df = erin_1g_sel_detvar_df.filter(pl.col("pandora_crtveto") == 0)

make_histogram_plot(
    pred_and_data_sel_df=erin_1g_crt_df,
    bins=proton_bins,
    var=proton_count_var,
    display_var=proton_count_display_var,
    title="Erin Inclusive 1g Selection with CRT veto",
    detvar_df=erin_1g_crt_detvar_df,
    selname="erin_inclusive_1g_sel_crt",
    breakdown_type="erin_categories",
    ylim=[0, 50], yticks=np.linspace(0, 50, 11),
    **sys_kwargs,
)

## Blip Variable Distributions

Diagnostic plots to understand the blip variables before using them to split the selection.

In [ ]:
for blip_var in [
    "blip_backtrack_cones_n",
    "blip_no_shower_cone_no_backtrack_cones_n",
    "blip_sphere_n",
    "blip_no_shower_cone_n",
]:
    make_histogram_plot(
        pred_and_data_sel_df=erin_1g_crt_df,
        bins=np.linspace(0, 20, 21),
        var=blip_var,
        display_var=blip_var,
        title="Erin Inclusive 1g Selection with CRT veto",
        detvar_df=erin_1g_crt_detvar_df,
        selname="erin_inclusive_1g_sel_crt",
        breakdown_type="erin_Np0p_categories",
        **sys_kwargs,
    )

## Np/0p Split: Protons + Backtrack Blips

In [ ]:
make_histogram_plot(
    pred_and_data_sel_df=erin_1g_crt_df,
    bins=proton_bins,
    var=proton_blip_var,
    display_var=proton_blip_display_var,
    title="Erin Inclusive 1g Selection with CRT veto",
    detvar_df=erin_1g_crt_detvar_df,
    selname="erin_inclusive_1g_sel_crt",
    breakdown_type="erin_Np0p_categories",
    ylim=[0, 50], yticks=np.linspace(0, 50, 11),
    **sys_kwargs,
)

## Nn/0n Blip Split

Split events using `blip_no_shower_cone_no_backtrack_cones_n` and `sumE`:
- **0n**: n ≤ 2 AND sumE ≤ 11
- **Nn**: n > 2 OR sumE > 11

In [ ]:
erin_1g_crt_Nn_df = erin_1g_crt_df.filter(Nn_expr)
erin_1g_crt_0n_df = erin_1g_crt_df.filter(zero_n_expr)
erin_1g_crt_Nn_detvar_df = erin_1g_crt_detvar_df.filter(Nn_expr)
erin_1g_crt_0n_detvar_df = erin_1g_crt_detvar_df.filter(zero_n_expr)

for df, detvar_df_split, title_suffix, selname in [
    (erin_1g_crt_Nn_df, erin_1g_crt_Nn_detvar_df, "Nn blip cut", "erin_inclusive_1g_sel_crt_Nn"),
    (erin_1g_crt_0n_df, erin_1g_crt_0n_detvar_df, "0n blip cut", "erin_inclusive_1g_sel_crt_0n"),
]:
    make_histogram_plot(
        pred_and_data_sel_df=df,
        bins=proton_bins,
        var=proton_blip_var,
        display_var=proton_blip_display_var,
        title=f"Erin Inclusive 1g Selection with CRT veto and {title_suffix}",
        detvar_df=detvar_df_split,
        selname=selname,
        breakdown_type="erin_Np0pNn0n_categories",
        **sys_kwargs,
    )

# NC Pi0 Sideband

In [ ]:
erin_ncpi0_sel_df = presel_merged_df.filter(pl.col("erin_nc_pi0_sideband_flag") == 1)
erin_ncpi0_crt_df = erin_ncpi0_sel_df.filter(pl.col("pandora_crtveto") == 0)
erin_ncpi0_crt_Nn_df = erin_ncpi0_crt_df.filter(Nn_expr)
erin_ncpi0_crt_0n_df = erin_ncpi0_crt_df.filter(zero_n_expr)

erin_ncpi0_sel_detvar_df = detvar_df.filter(pl.col("erin_nc_pi0_sideband_flag") == 1)
erin_ncpi0_crt_Nn_detvar_df = erin_ncpi0_sel_detvar_df.filter(Nn_expr)
erin_ncpi0_crt_0n_detvar_df = erin_ncpi0_sel_detvar_df.filter(zero_n_expr)

In [ ]:
for df, detvar_df_split, title_suffix, selname in [
    (erin_ncpi0_crt_Nn_df, erin_ncpi0_crt_Nn_detvar_df, "Nn blip cut", "erin_ncpi0_sel_crt_Nn"),
    (erin_ncpi0_crt_0n_df, erin_ncpi0_crt_0n_detvar_df, "0n blip cut", "erin_ncpi0_sel_crt_0n"),
]:
    make_histogram_plot(
        pred_and_data_sel_df=df,
        bins=proton_bins,
        var=proton_blip_var,
        display_var=proton_blip_display_var,
        title=f"Erin NC Pi0 Sideband with CRT veto and {title_suffix}",
        detvar_df=detvar_df_split,
        selname=selname,
        breakdown_type="erin_categories",
        ylim=[0, 240], yticks=np.linspace(0, 240, 13),
        **sys_kwargs,
    )

In [ ]:
for df, detvar_df_split, title_suffix, selname in [
    (erin_ncpi0_crt_Nn_df, erin_ncpi0_crt_Nn_detvar_df, "Nn blip cut", "erin_ncpi0_sel_crt_Nn"),
    (erin_ncpi0_crt_0n_df, erin_ncpi0_crt_0n_detvar_df, "0n blip cut", "erin_ncpi0_sel_crt_0n"),
]:
    make_histogram_plot(
        pred_and_data_sel_df=df,
        bins=proton_bins,
        var=proton_blip_var,
        display_var=proton_blip_display_var,
        title=f"Erin NC Pi0 Sideband with CRT veto and {title_suffix}",
        detvar_df=detvar_df_split,
        selname=selname,
        breakdown_type="erin_Np0pNn0n_pi0_categories",
        **sys_kwargs,
    )